# MMSep Module 2 — Kaggle LLaVA v1.5 7B Baseline smoke test

This notebook runs one **non-formal** image inference. Before running it:

1. Open **Settings**, enable **GPU** and **Internet**.
2. Use **Add Input / Upload** to attach `mmsep_kaggle_source.zip`.
3. Attach exactly one ordinary PNG/JPG/JPEG/WebP test image.

The model and Python environment stay on temporary scratch storage. Only the small result files are written to `/kaggle/working/mmsep_results`. No API key or GitHub push is used.

In [ ]:
# 1. Hardware, Internet, and scratch-disk gates.
import shutil, subprocess, urllib.request
from pathlib import Path

assert shutil.which('nvidia-smi'), 'No NVIDIA GPU is attached. Enable GPU in Kaggle Settings.'
query = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'],
    check=True, capture_output=True, text=True,
).stdout.strip().splitlines()[0]
gpu_name, total_text, free_text = [part.strip() for part in query.split(',')]
total_mib, free_mib = int(total_text), int(free_text)
assert total_mib >= 12 * 1024, f'GPU VRAM is too small: {total_mib} MiB'
assert free_mib >= 10 * 1024, 'Restart the session: less than 10 GB GPU memory is free.'
try:
    with urllib.request.urlopen('https://pypi.org/simple/', timeout=15) as response:
        assert response.status == 200
except Exception as exc:
    raise AssertionError('Internet is disabled or unavailable in Kaggle Settings.') from exc
scratch_candidates = [path for path in (Path('/kaggle/temp'), Path('/tmp')) if path.exists()]
SCRATCH = max(scratch_candidates, key=lambda path: shutil.disk_usage(path).free)
scratch_free_gib = shutil.disk_usage(SCRATCH).free / (1024 ** 3)
assert scratch_free_gib >= 28, f'At least 28 GiB scratch space is required; found {scratch_free_gib:.1f} GiB.'
RUNTIME_ROOT = SCRATCH / 'mmsep_runtime'
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
print({'gpu': gpu_name, 'total_mib': total_mib, 'free_mib': free_mib, 'scratch': str(SCRATCH), 'scratch_free_gib': round(scratch_free_gib, 1)})

In [ ]:
# 2. Locate and safely stage the reviewed project source input.
import os, shutil, zipfile
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
PROJECT_ROOT = Path('/kaggle/working/mmsep_project')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
archives = list(INPUT_ROOT.rglob('mmsep_kaggle_source.zip'))
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        root = PROJECT_ROOT.resolve()
        for member in archive.infolist():
            target = (PROJECT_ROOT / member.filename).resolve()
            assert os.path.commonpath([root, target]) == str(root), f'Unsafe ZIP member: {member.filename}'
        archive.extractall(PROJECT_ROOT)
elif len(archives) == 0:
    # Kaggle may unpack uploaded ZIP files automatically.
    markers = list(INPUT_ROOT.rglob('src/mmsep_testkit'))
    assert len(markers) == 1, 'Cannot uniquely locate the uploaded project source.'
    unpacked_root = markers[0].parents[1]
    for name in ('src', 'configs', 'requirements', 'tests'):
        shutil.copytree(unpacked_root / name, PROJECT_ROOT / name, dirs_exist_ok=True)
else:
    raise AssertionError('Attach exactly one mmsep_kaggle_source.zip input.')
assert (PROJECT_ROOT / 'src/mmsep_testkit').is_dir(), 'Project source structure is invalid.'
print('Project source:', PROJECT_ROOT)

In [ ]:
# 3. Build an isolated Python 3.10/LLaVA environment on scratch storage.
import os, subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv', 'modelscope-hub'], check=True)
UV_ENV = os.environ.copy()
UV_ENV.pop('UV_SYSTEM_PYTHON', None)  # Suppress Kaggle's harmless --system warning.
VENV = RUNTIME_ROOT / 'venv310'
VENV_PY = VENV / 'bin/python'
subprocess.run(['uv', 'venv', '--clear', '--python', '3.10', str(VENV)], check=True, env=UV_ENV)
source_requirements = PROJECT_ROOT / 'requirements/llava-cloud-linux.txt'
runtime_requirements = RUNTIME_ROOT / 'llava-cloud-runtime.txt'
filtered_lines = [
    line for line in source_requirements.read_text(encoding='utf-8').splitlines()
    if not line.startswith(('--extra-index-url', 'torch==', 'torchvision==', 'bitsandbytes=='))
]
runtime_requirements.write_text('\n'.join(filtered_lines) + '\n', encoding='utf-8')
subprocess.run([
    'uv', 'pip', 'install', '--python', str(VENV_PY),
    '--default-index', 'https://download.pytorch.org/whl/cu121',
    'torch==2.4.1+cu121', 'torchvision==0.19.1+cu121',
], check=True, env=UV_ENV)
subprocess.run([
    'uv', 'pip', 'install', '--python', str(VENV_PY),
    '--default-index', 'https://pypi.org/simple', 'bitsandbytes==0.48.2', '-r', str(runtime_requirements),
], check=True, env=UV_ENV)
subprocess.run(['uv', 'pip', 'check', '--python', str(VENV_PY)], check=True, env=UV_ENV)
site_packages = VENV / 'lib/python3.10/site-packages'
cuda_library_dirs = [str(path) for path in (site_packages / 'nvidia').glob('*/lib') if path.is_dir()]
if Path('/usr/local/cuda/lib64').is_dir():
    cuda_library_dirs.append('/usr/local/cuda/lib64')
existing_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
if existing_ld_path:
    cuda_library_dirs.append(existing_ld_path)
os.environ['LD_LIBRARY_PATH'] = ':'.join(cuda_library_dirs)
UV_ENV['LD_LIBRARY_PATH'] = os.environ['LD_LIBRARY_PATH']
LLAVA_SOURCE = RUNTIME_ROOT / 'LLaVA-v1.2.0'
if not LLAVA_SOURCE.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'v1.2.0',
        'https://github.com/haotian-liu/LLaVA.git', str(LLAVA_SOURCE),
    ], check=True)
subprocess.run([
    'uv', 'pip', 'install', '--python', str(VENV_PY),
    '--no-deps', '-e', str(LLAVA_SOURCE),
], check=True, env=UV_ENV)
subprocess.run([
    str(VENV_PY), '-c',
    "import importlib.metadata as m, torch, transformers, llava; print('torch/cuda/transformers/bnb=', torch.__version__, torch.version.cuda, transformers.__version__, m.version('bitsandbytes')); print('cuda=', torch.cuda.is_available(), torch.cuda.get_device_name(0))",
], check=True, env=UV_ENV)

In [ ]:
# 4. Download the checkpoint and CLIP vision tower to non-persistent scratch storage.
import subprocess

MODEL_ROOT = RUNTIME_ROOT / 'models'
MODEL_PATH = MODEL_ROOT / 'llava-v1.5-7b'
VISION_PATH = MODEL_ROOT / 'clip-vit-large-patch14-336'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
if not (MODEL_PATH / 'pytorch_model.bin.index.json').is_file():
    subprocess.run([
        'ms-hub', 'download', 'huangjianuo/llava-v1.5-7b',
        '--revision', 'master', '--local-dir', str(MODEL_PATH),
    ], check=True)
if not (VISION_PATH / 'config.json').is_file():
    subprocess.run([
        'ms-hub', 'download', 'openai-mirror/clip-vit-large-patch14-336',
        '--revision', 'master', '--local-dir', str(VISION_PATH),
    ], check=True)
print('Model:', MODEL_PATH)
print('Vision tower:', VISION_PATH)

In [ ]:
# 5. Run unit tests and the mandatory model preflight.
import os, subprocess

BASE_CONFIG = PROJECT_ROOT / 'configs/experiments/baseline.llava.example.json'
RUN_ENV = os.environ.copy()
RUN_ENV.update({
    'PYTHONPATH': str(PROJECT_ROOT / 'src'),
    'LLAVA_MODEL_PATH': str(MODEL_PATH),
    'LLAVA_VISION_TOWER_PATH': str(VISION_PATH),
    'LLAVA_CODE_PATH': str(LLAVA_SOURCE),
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'TOKENIZERS_PARALLELISM': 'false',
})
subprocess.run([
    str(VENV_PY), '-m', 'unittest', 'discover',
    '-s', str(PROJECT_ROOT / 'tests/unit'), '-v',
], env=RUN_ENV, check=True)
preflight = subprocess.run([
    str(VENV_PY), '-m', 'mmsep_testkit.preflight',
    '--config', str(BASE_CONFIG),
], env=RUN_ENV, text=True)
assert preflight.returncode == 0, 'Preflight failed. Do not continue until every failed check is fixed.'

In [ ]:
# 6. Select the attached image and create a non-formal 32-token smoke case.
import json
from pathlib import Path

IMAGE_PATH_OVERRIDE = ''  # Set an absolute /kaggle/input/... path only if multiple images are attached.
if IMAGE_PATH_OVERRIDE:
    IMAGE_PATH = Path(IMAGE_PATH_OVERRIDE)
else:
    images = [path for path in INPUT_ROOT.rglob('*') if path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}]
    assert len(images) == 1, f'Expected exactly one image input, found {len(images)}. Set IMAGE_PATH_OVERRIDE.'
    IMAGE_PATH = images[0]
assert IMAGE_PATH.is_file(), f'Image not found: {IMAGE_PATH}'
CASE_DIR = Path('/kaggle/working/mmsep_cases')
CASE_DIR.mkdir(parents=True, exist_ok=True)
CASE_PATH = CASE_DIR / 'baseline_smoke.jsonl'
case = {
    'case_id': 'M2-FUN-SMOKE-001',
    'title': 'Single-image Baseline smoke test',
    'dimension': 'functional',
    'input': {'text': 'Describe the main objects and scene in this image briefly.', 'image_ref': str(IMAGE_PATH)},
    'formal': False,
    'metadata': {'source': 'manual-kaggle-smoke'},
}
CASE_PATH.write_text(json.dumps(case, ensure_ascii=False) + '\n', encoding='utf-8')
smoke_config = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))
smoke_config['generation']['max_new_tokens'] = 32
smoke_config['experiment']['seeds'] = [42]
smoke_config['experiment']['repetitions'] = 1
SMOKE_CONFIG = CASE_DIR / 'baseline.smoke.local.json'
SMOKE_CONFIG.write_text(json.dumps(smoke_config, ensure_ascii=False, indent=2), encoding='utf-8')
print('Image:', IMAGE_PATH)
print('Case:', CASE_PATH)

In [ ]:
# 7. Explicit execution gate: load the 7B model and run one inference.
import json, subprocess
from pathlib import Path

RESULT_DIR = Path('/kaggle/working/mmsep_results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_PATH = RESULT_DIR / 'baseline-smoke.jsonl'
run = subprocess.run([
    str(VENV_PY), '-m', 'mmsep_testkit.runner',
    '--config', str(SMOKE_CONFIG),
    '--cases', str(CASE_PATH),
    '--output', str(RESULT_PATH),
], env=RUN_ENV, text=True)
assert run.returncode == 0, 'Baseline inference failed; preserve the full cell output for diagnosis.'
result = json.loads(RESULT_PATH.read_text(encoding='utf-8').splitlines()[0])
assert result.get('error') is None, f"Baseline inference returned an error: {result.get('error')}"
assert result.get('metrics', {}).get('output_tokens', 0) > 0, 'Baseline generated zero tokens.'
assert result.get('output_text', '').strip(), 'Baseline generated empty text.'
print(json.dumps({
    'case_id': result['case_id'],
    'output_text': result['output_text'],
    'duration_ms': result['duration_ms'],
    'metrics': result['metrics'],
    'result_path': str(RESULT_PATH),
}, ensure_ascii=False, indent=2))

In [ ]:
# 8. Save a compact runtime manifest next to the result for later evidence review.
from datetime import datetime, timezone
import json, subprocess

versions = subprocess.run([
    str(VENV_PY), '-c',
    "import importlib.metadata as m, torch, transformers; print(torch.__version__); print(transformers.__version__); print(m.version('bitsandbytes'))",
], capture_output=True, text=True, check=True).stdout.strip().splitlines()
manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'platform': 'kaggle',
    'gpu': gpu_name,
    'gpu_total_mib': total_mib,
    'python': '3.10',
    'torch': versions[0],
    'transformers': versions[1],
    'bitsandbytes': versions[2],
    'formal': False,
}
manifest_path = RESULT_DIR / 'baseline-smoke.runtime.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved Kaggle outputs:')
for path in sorted(RESULT_DIR.iterdir()):
    print(' -', path)